# Congress Momentum + QUIVER_OPEN — Portfolio A Backtest

**Winner strategy** confirmed in `V2_02_basic_strategies.ipynb` §7–8.  
Do not modify this notebook for exploration — use V2_02 for that.

**Setup:**
- Universe: US Congressional **Purchase** disclosures on QuiverQuant
- Filter: `Annualized_Traded_To_File > MOMENTUM_THRESHOLD` (stock already moving before Quiver published)
- Entry: Buy at **open** on Quiver upload day
- Exit: Sell at **close** on the same day
- Quiver delay cap: ≤ `MAX_QUIVER_DELAY` days from SEC filing

**Confirmed results (V2_02 §8, 2016–2025):**
- $100K → $155,545 | **11.99% annualized** | Max drawdown **-4.63%** | Positive **9/10 years**
- QUIVER_OPEN outperforms all other timing variants on the filtered set

> ⚠️ Live execution requires intraday Quiver API polling. Trades uploaded after market close are not executable same-day.

## 1. Setup

In [ ]:
import sys
sys.path.insert(0, "../../")  # algo_trading/shared/

import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from _shared.significance import full_significance_report, print_significance_report

In [ ]:
# =============================================================================
# CONFIGURATION — winner from V2_02
# =============================================================================
DATA_PKL           = r"C:\Users\danie\Desktop\INVESTMENT PROJECT\Congress Things\_ARCHIVE\final_data_MEGA.pkl"
MOMENTUM_THRESHOLD = 1.5    # Annualized_Traded_To_File > this value
MAX_QUIVER_DELAY   = 3      # max days from SEC filing to Quiver upload
INITIAL_CAPITAL    = 100_000
MAX_PCT_PER_TRADE  = 0.10   # max fraction of capital per trade
SAVE_NAME          = "congress_momentum_quiver_open" 

## 2. Load Data

In [ ]:
with open(DATA_PKL, "rb") as f:
    data = pickle.load(f)

final_df   = data["congresist_df_dict"]
stock_dict = data["stock_dict"]

print(f"Trades: {len(final_df):,}  |  Tickers: {len(stock_dict):,}  |  Politicians: {final_df['Name'].nunique()}")
print(f"Range:  {final_df['Filed'].min()} → {final_df['Filed'].max()}")

## 3. Compute Returns + Apply Winner Filter

Only computes what's needed for the winning strategy: `return_1d_QUIVER_OPEN` on momentum-filtered purchases.
No other variants computed here — see `V2_02_basic_strategies.ipynb` for the full exploration.

In [ ]:
df = final_df.copy()
df['Filed']              = pd.to_datetime(df['Filed'])
df['Quiver_Upload_Time'] = pd.to_datetime(df['Quiver_Upload_Time'])
df['Quiver_Upload_Date'] = df['Quiver_Upload_Time'].dt.normalize()

# ── Quiver delay filter ───────────────────────────────────────────────────────
# Use Quiver gap only for filtering — do NOT overwrite native days_to_file column
# (native days_to_file = trade date → SEC filing, mean ~46 days; Quiver gap = 1-3 days)
quiver_gap = (df['Quiver_Upload_Date'] - df['Filed']).dt.days.abs()
df = df[quiver_gap <= MAX_QUIVER_DELAY].copy()

# ── Momentum filter (uses native days_to_file: trade date → SEC filing) ───────
df['days_to_file_nz'] = df['days_to_file'].replace(0, np.nan)
df['Annualized_Traded_To_File'] = (1 + df['Return_Traded_To_File']) ** (365 / df['days_to_file_nz']) - 1

df_filtered = df[
    (df['Transaction'] == 'Purchase') &
    (df['Annualized_Traded_To_File'] > MOMENTUM_THRESHOLD)
].copy()

print(f"After delay filter:    {len(df):,} rows")
print(f"After momentum filter: {len(df_filtered):,} rows  ({len(df_filtered)/len(df)*100:.1f}%)")

In [ ]:
# ── Compute QUIVER_OPEN return ────────────────────────────────────────────────
def get_open_to_close(stock_dict, ticker, date):
    """Return: close/open - 1 on `date` for `ticker`. Uses first available day >= date."""    if ticker not in stock_dict:
        return np.nan
    s = stock_dict[ticker].copy()
    s['timestamp'] = pd.to_datetime(s['timestamp']).dt.tz_localize(None).dt.normalize()
    s = s.sort_values('timestamp')
    row = s[s['timestamp'] >= date]
    if row.empty or row.iloc[0]['open'] == 0:
        return np.nan
    return row.iloc[0]['close'] / row.iloc[0]['open'] - 1

print("Computing QUIVER_OPEN returns (first run takes a few minutes)...")
df_filtered = df_filtered.copy()
df_filtered['return_1d_QUIVER_OPEN'] = df_filtered.apply(
    lambda r: get_open_to_close(stock_dict, r['Ticker'], r['Quiver_Upload_Date']), axis=1
)
df_filtered = df_filtered.dropna(subset=['return_1d_QUIVER_OPEN'])
print(f"Trades with valid returns: {len(df_filtered):,}")
print(f"Mean return: {df_filtered['return_1d_QUIVER_OPEN'].mean()*100:+.3f}%")

## 4. Statistical Significance

In [ ]:
trades_sig = df_filtered[['return_1d_QUIVER_OPEN']].copy()
trades_sig['net_pnl']       = trades_sig['return_1d_QUIVER_OPEN'] * INITIAL_CAPITAL * MAX_PCT_PER_TRADE
trades_sig['equity_before'] = INITIAL_CAPITAL

report = full_significance_report(trades_sig, strategy_name="Congress Momentum QUIVER_OPEN")
print_significance_report(report)

## 5. Capital Simulation

In [ ]:
df_filtered['Filed'] = pd.to_datetime(df_filtered['Filed'])
df_sim = df_filtered.sort_values('Filed').reset_index(drop=True)

capital = INITIAL_CAPITAL
equity_ts = []

for _, row in df_sim.iterrows():
    amount = capital * MAX_PCT_PER_TRADE
    pnl    = amount * row['return_1d_QUIVER_OPEN']
    capital += pnl
    equity_ts.append((row['Filed'], capital))

equity = pd.Series(dict(equity_ts)).sort_index()
equity.index = pd.to_datetime(equity.index)

total_return = capital / INITIAL_CAPITAL - 1
total_days   = (equity.index[-1] - equity.index[0]).days
ann_return   = (capital / INITIAL_CAPITAL) ** (365 / total_days) - 1

daily = equity.resample('D').last().ffill()
dr    = daily.pct_change().dropna()
sharpe = dr.mean() / dr.std() * np.sqrt(252) if dr.std() > 0 else 0

peak   = equity.cummax()
max_dd = ((equity - peak) / peak).min()

eq_year = equity.resample('YE').last()
eq_prev = eq_year.shift(1).fillna(INITIAL_CAPITAL)
per_year = (eq_year.values / eq_prev.values - 1)

print(f"Initial capital:    ${INITIAL_CAPITAL:>12,.2f}")
print(f"Final capital:      ${capital:>12,.2f}")
print(f"Total return:       {total_return:>12.2%}")
print(f"Annualized return:  {ann_return:>12.2%}")
print(f"Sharpe ratio:       {sharpe:>12.2f}")
print(f"Max drawdown:       {max_dd:>12.2%}")
print(f"N trades:           {len(df_sim):>12,}")
print()
print("Per-year returns:")
for yr, r in zip(eq_year.index.year, per_year):
    bar = '█' * max(0, int(r * 200))
    neg = '░' * max(0, int(-r * 200))
    print(f"  {yr}:  {r:+.2%}  {'  ' + bar if r >= 0 else neg}")

## 6. Equity Curve + Drawdown

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), gridspec_kw={'height_ratios': [3, 1]})

ax1.plot(equity.index, equity.values, linewidth=1.5, color='steelblue',
         label=f"QUIVER_OPEN momentum — {ann_return:+.1%} ann | Sh {sharpe:.2f} | DD {max_dd:.1%}")
ax1.axhline(INITIAL_CAPITAL, color='black', linestyle=':', alpha=0.4, label='Start')
ax1.set_ylabel('Portfolio Value ($)')
ax1.set_title('Congress Momentum + QUIVER_OPEN — Portfolio A')
ax1.legend()
ax1.grid(alpha=0.3)

dd_series = (equity - equity.cummax()) / equity.cummax()
ax2.fill_between(dd_series.index, dd_series.values, 0, alpha=0.5, color='red')
ax2.set_ylabel('Drawdown')
ax2.set_ylim(min(-0.20, dd_series.min() * 1.2), 0.05)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Save Trades

In [ ]:
import os
os.makedirs("results", exist_ok=True)

trades_out = df_filtered.copy().rename(columns={
    'Quiver_Upload_Date': 'entry_time',
    'Ticker':             'instrument',
    'return_1d_QUIVER_OPEN': 'pct_return_gross',
})
trades_out['exit_time']   = trades_out['entry_time']
trades_out['direction']   = 'long'
trades_out['entry_price'] = np.nan
trades_out['exit_price']  = np.nan
trades_out['exit_reason'] = 'eod_close'
trades_out['stop_price']  = np.nan

std_cols = ['entry_time','exit_time','direction','instrument',
            'entry_price','exit_price','pct_return_gross','exit_reason','stop_price']
path = f"results/{SAVE_NAME}_trades.csv"
trades_out[std_cols].to_csv(path, index=False)
print(f"Saved {len(trades_out):,} trades → {path}")
print(f"Sharpe: {sharpe:.2f}  |  Ann return: {ann_return:.2%}  |  Max DD: {max_dd:.2%}")

## 8. Conclusions

**Strategy: Congress Momentum + QUIVER_OPEN — Portfolio A**

| Metric | Value |
|---|---|
| Annualized return | see §5 output |
| Sharpe ratio | see §5 output |
| Max drawdown | see §5 output |
| Positive years | see §5 output |
| N trades (backtest) | see §5 output |

**Go/No-go criteria (fill in after running):**
- [ ] Sharpe > 0.8
- [ ] Annualized return > 10%
- [ ] Statistical significance: 2/3 tests pass (see §4)
- [ ] Max drawdown < 15%
- [ ] Positive in ≥ 7/10 years

**Next:** `congress_momentum_Implementation.ipynb` — poll Quiver API intraday, place market-open orders via Alpaca, sell at close.

In [ ]:
# ── §9. Save outputs for Portfolio_A_Analysis.ipynb ──────────────────────────
import os, json as _json

os.makedirs(f'results/{SAVE_NAME}_daily_equity', exist_ok=True)

# Daily equity CSV
daily = equity.resample('B').last().ffill()
daily.rename('equity').to_frame().rename_axis('date').to_csv(
    f'results/{SAVE_NAME}_daily_equity/quiver_open_10pct_1x.csv'
)

# Implementations JSON
impl = {
    'quiver_open_10pct_1x': {
        'total_return': round((capital / INITIAL_CAPITAL - 1) * 100, 1),
        'cagr':   round(ann_return * 100, 2),
        'sharpe': round(sharpe, 2),
        'max_dd': round(max_dd * 100, 1),
        'label':  'QUIVER_OPEN 10% per trade (1x)',
    }
}
with open(f'results/{SAVE_NAME}_implementations.json', 'w') as f:
    _json.dump(impl, f, indent=2)

print(f'Saved results/{SAVE_NAME}_implementations.json')
print(f'Saved results/{SAVE_NAME}_daily_equity/quiver_open_10pct_1x.csv')
